# CloudCompute: обучение модели
Универсальный запуск MobileNet, EfficientNet или ViT с локальным recovery и экспортом результата.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
MODEL = "vit"  # mobilenet | efficientnet | vit
QUICK_RUN = False
RESUME_TRAINING = True
REQUIRE_EXISTING_RECOVERY = True  # защита для продолжения уже начатого запуска
RUN_TESTS = False
RUN_CALIBRATION = True
PROMOTE_CHAMPION = False  # True, только если STATE_DIR уже содержит registry
TRAIN_BATCH_SIZE = 16  # ViT: 16, при OOM: 8; None = значение config
VALIDATION_BATCH_SIZE = 32  # ViT: 32, при OOM: 16
NUM_WORKERS = 2
REPO_DIR = "/root/text-orientation-classification"
STATE_DIR = "/root/text-orientation-state"

In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from datetime import datetime, timezone
model_options = {
    "mobilenet": ("configs/baseline.yaml", "mobilenet_v3_large"),
    "efficientnet": ("configs/efficientnet_b0.yaml", "efficientnet_b0"),
    "vit": ("configs/vit_b_16.yaml", "vit_b_16"),
}
if MODEL not in model_options:
    raise ValueError(f"Unknown MODEL={MODEL!r}")
config, model_name = model_options[MODEL]
mode = "quick" if QUICK_RUN else "full"
run_name = f"cloud_{model_name}_{mode}"
run_dir = Path("artifacts/experiments") / run_name
recovery_dir = Path(STATE_DIR) / "training/recovery" / model_name / mode
recovery_dir.mkdir(parents=True, exist_ok=True)
if RESUME_TRAINING and REQUIRE_EXISTING_RECOVERY and not (recovery_dir / "last.pt").is_file():
    raise FileNotFoundError(f"Upload recovery files first: {recovery_dir}")
command = [sys.executable, "-m", "scripts.train", "--config", config, "--run-name", run_name, "--recovery-dir", str(recovery_dir), "--num-workers", str(NUM_WORKERS)]
if TRAIN_BATCH_SIZE is not None: command += ["--batch-size", str(TRAIN_BATCH_SIZE)]
if VALIDATION_BATCH_SIZE is not None: command += ["--validation-batch-size", str(VALIDATION_BATCH_SIZE)]
if RESUME_TRAINING: command.append("--resume")
if QUICK_RUN: command += ["--train-base-samples", "2048", "--validation-base-samples", "512", "--frozen-epochs", "1", "--finetune-epochs", "2"]
print("Recovery:", recovery_dir)
subprocess.run(command, check=True)
if RUN_CALIBRATION and not QUICK_RUN:
    subprocess.run([sys.executable, "-m", "scripts.calibrate", "--run-dir", str(run_dir), "--config", config], check=True)
if PROMOTE_CHAMPION and not QUICK_RUN:
    subprocess.run([sys.executable, "-m", "scripts.promote_champion", "--run-dir", str(run_dir), "--registry-dir", str(Path(STATE_DIR) / "registry")], check=True)
exports = Path(STATE_DIR) / "training/runs" / model_name / mode
exports.mkdir(parents=True, exist_ok=True)
archive = exports / f"{run_name}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
subprocess.run(["zip", "-qr", str(archive) + ".zip", str(run_dir)], check=True)
print("Result:", str(archive) + ".zip")